# ResNet-18 на CIFAR-10

Репликация ResNet-18 (He et al., 2016) с двумя целями:

1. **Проверить блоки библиотеки.** `ResidualBlock` из `spartan_torch` собирается в
   ResNet-18, веса копируются из `torchvision`, форварды сравниваются поэлементно.
2. **Референс-пайплайн классификации изображений.** Трейн через pytorch-lightning,
   логгирование в MLflow, графики inline через коллбэк.

Запуск: `uv sync --extra dev --extra experiments`, затем `uv run jupyter lab` и открыть
этот ноутбук. Cwd ноутбука = его папка, поэтому данные и чекпойнты лягут рядом с ним.


In [ ]:
# === Setup: среда + зависимости (локально / devcontainer / Colab) ===
import sys

IN_COLAB = "google.colab" in sys.modules
MLFLOW_ENABLED = not IN_COLAB  # в Colab локального MLflow-сервера нет

if IN_COLAB:
    import subprocess
    from pathlib import Path
    PROJECT_ROOT = Path("/content/spartan-torch")
    if not (PROJECT_ROOT / ".git").exists():
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/mievst/spartan-torch.git", str(PROJECT_ROOT)],
            check=True,
        )
    # репо-модули экспериментов (tinyllama/*.py, vit/vision_transformer.py) + исходники lib
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
    sys.path.insert(0, str(PROJECT_ROOT / "experiments"))
    # Colab приходит со своим numpy/scipy; наш -e ресолв мог рассогласовать их.
    # Апгрейдим пару вместе, чтобы они совпали (иначе datasets -> scipy падает
    # на numpy._core._multiarray_umath._blas_supports_fpe).
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-U",
         "--upgrade-strategy", "eager", "numpy", "scipy"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e",
         f"{PROJECT_ROOT}[experiments,dev]"],
        check=True,
    )
else:
    PROJECT_ROOT = None

print(f"IN_COLAB={IN_COLAB} | MLFLOW_ENABLED={MLFLOW_ENABLED} | PROJECT_ROOT={PROJECT_ROOT} | python={sys.version.split()[0]}")


## 0. Конфигурация

Пути и гиперпараметры — всё в одной ячейке. `data/` и `checkpoints/` уже
в `.gitignore`, в git уходит только ноутбук.

In [1]:
from pathlib import Path

import torch

ROOT = (PROJECT_ROOT / "experiments/image_classification/resnet18") if IN_COLAB else Path.cwd()
DATA_DIR = ROOT / "data"
CKPT_DIR = ROOT / "checkpoints"
DATA_DIR.mkdir(exist_ok=True)
CKPT_DIR.mkdir(exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"cwd: {ROOT} | device: {DEVICE}")

# --- гиперпараметры ---
EPOCHS = 65
BATCH_SIZE = 128
LR = 0.1
MOMENTUM = 0.9
WEIGHT_DECAY = 5e-4
WARMUP_EPOCHS = 5
MIN_LR = 1e-4
NUM_WORKERS = 0  # Windows-safe
SEED = 0
PROFILER = "simple"  # None | "simple" | "advanced" | "pytorch"

torch.manual_seed(SEED)

# --- MLflow ---
MLFLOW_TRACKING_URI = "http://localhost:5000"
MLFLOW_EXPERIMENT_NAME = "resnet18-cifar10"

torch.set_float32_matmul_precision("medium")


cwd: a:\projects\spartan-torch\experiments\image_classification\resnet18 | device: cuda


## 1. Сборка ResNet-18

Сеть собирается из `spartan_torch.ResidualBlock` и примитивов `torch.nn`.
Сама сеть — код эксперимента (AGENTS.md: реимплементация стандартных моделей
в библиотеку не входит), в библиотеку попадает только переиспользуемый блок.

In [2]:
from collections import OrderedDict

from torch import nn

from spartan_torch import ResidualBlock


def make_resnet18(num_classes: int) -> nn.Module:
    """ResNet-18, структурно идентичный torchvision."""

    def stage(in_c: int, out_c: int, num_blocks: int, stride: int = 1) -> nn.Sequential:
        blocks = OrderedDict()
        blocks["0"] = ResidualBlock(in_c, out_c, stride=stride)
        for i in range(1, num_blocks):
            blocks[str(i)] = ResidualBlock(out_c, out_c)
        return nn.Sequential(blocks)

    return nn.Sequential(OrderedDict([
        ("conv1", nn.Conv2d(3, 64, 7, stride=2, padding=3, bias=False)),
        ("bn1", nn.BatchNorm2d(64)),
        ("relu", nn.ReLU(inplace=True)),
        ("maxpool", nn.MaxPool2d(kernel_size=3, stride=2, padding=1)),
        ("layer1", stage(64, 64, 2)),
        ("layer2", stage(64, 128, 2, stride=2)),
        ("layer3", stage(128, 256, 2, stride=2)),
        ("layer4", stage(256, 512, 2, stride=2)),
        ("avgpool", nn.AdaptiveAvgPool2d((1, 1))),
        ("flatten", nn.Flatten(1)),
        ("fc", nn.Linear(512, num_classes)),
    ]))


## 2. Верификация блоков против torchvision

Ключевой шаг проверки библиотеки: `ResidualBlock` повторяет структуру
`BasicBlock` torchvision 1-в-1 (`conv1`/`bn1`/`conv2`/`bn2`/`downsample`),
поэтому `state_dict` грузится напрямую. Если форварды совпадут поэлементно —
блоки эквивалентны.

In [3]:
import torchvision

vision = torchvision.models.resnet18(weights=None)
mine = make_resnet18(num_classes=1000)

missing, unexpected = mine.load_state_dict(vision.state_dict(), strict=False)
assert not missing and not unexpected, f"state_dict mismatch: {missing} {unexpected}"
print(f"state_dict keys matched: {len(vision.state_dict())}")

vision.eval()
mine.eval()
x = torch.randn(4, 3, 32, 32)
with torch.no_grad():
    y_vision = vision(x)
    y_mine = mine(x)

max_diff = (y_mine - y_vision).abs().max().item()
assert torch.allclose(y_mine, y_vision, atol=1e-6), "FORWARD MISMATCH"
print(f"forward identical | max abs diff = {max_diff:.2e}")

n_vision = sum(p.numel() for p in vision.parameters())
n_mine = sum(p.numel() for p in mine.parameters())
print(f"params | torchvision: {n_vision:,} | ours: {n_mine:,}")


state_dict keys matched: 122
forward identical | max abs diff = 0.00e+00
params | torchvision: 11,689,512 | ours: 11,689,512


## 3. Данные: CIFAR-10

`CIFAR10DataModule` качает данные в `data/` папки эксперимента — полная
изоляция между экспериментами.

In [4]:
import lightning as L
import torchvision.transforms as T
from pathlib import Path

from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR10


class CIFAR10DataModule(L.LightningDataModule):
    MEAN = (0.4914, 0.4822, 0.4465)
    STD = (0.2470, 0.2435, 0.2616)

    def __init__(self, data_dir, batch_size=128, num_workers=0):
        super().__init__()
        self.data_dir = Path(data_dir)
        self.batch_size = batch_size
        self.num_workers = num_workers

    def prepare_data(self):
        CIFAR10(self.data_dir, train=True, download=True)
        CIFAR10(self.data_dir, train=False, download=True)

    def setup(self, stage=None):
        train_tf = T.Compose([
            T.RandomCrop(32, padding=4),
            T.RandomHorizontalFlip(),
            T.ToTensor(),
            T.Normalize(self.MEAN, self.STD),
        ])
        val_tf = T.Compose([
            T.ToTensor(),
            T.Normalize(self.MEAN, self.STD),
        ])
        if stage in (None, "fit"):
            self.train_ds = CIFAR10(self.data_dir, train=True, transform=train_tf)
            self.val_ds = CIFAR10(self.data_dir, train=False, transform=val_tf)

    def train_dataloader(self):
        return DataLoader(
            self.train_ds, batch_size=self.batch_size, shuffle=True,
            num_workers=self.num_workers, persistent_workers=self.num_workers > 0,
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_ds, batch_size=self.batch_size, shuffle=False,
            num_workers=self.num_workers, persistent_workers=self.num_workers > 0,
        )


W0803 23:42:30.026000 36376 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


## 4. Обёртка Lightning

`configure_optimizers`: SGD + `WarmupScheduler` из библиотеки поверх
`CosineAnnealingLR` — заодно проверка ютилита в связке с lightning.

In [ ]:
import torch.nn.functional as F

from spartan_torch import WarmupScheduler
from torchmetrics import Accuracy


class ResNet18Lit(L.LightningModule):
    def __init__(self, num_classes=10, lr=0.1, momentum=0.9, weight_decay=5e-4,
                 warmup_epochs=5, epochs=60, min_lr=1e-4):
        super().__init__()
        self.save_hyperparameters()
        self.model = make_resnet18(num_classes)
        self.train_acc = Accuracy(task="multiclass", num_classes=num_classes)
        self.val_acc = Accuracy(task="multiclass", num_classes=num_classes)

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y)
        self.train_acc(logits, y)
        self.log("train_loss", loss, on_step=True, on_epoch=True, prog_bar=True)
        self.log("train_acc", self.train_acc, on_step=True, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y)
        self.val_acc(logits, y)
        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log("val_acc", self.val_acc, on_step=False, on_epoch=True, prog_bar=True)

    def configure_optimizers(self):
        optimizer = torch.optim.SGD(
            self.parameters(), lr=self.hparams.lr, momentum=self.hparams.momentum,
            weight_decay=self.hparams.weight_decay,
        )
        cosine = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=self.hparams.epochs - self.hparams.warmup_epochs,
            eta_min=self.hparams.min_lr,
        )
        scheduler = WarmupScheduler(
            optimizer, warmup=self.hparams.warmup_epochs,
            scheduler=cosine, min_lr=self.hparams.min_lr,
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {"scheduler": scheduler, "interval": "epoch", "frequency": 1},
        }


## 5. Коллбэки: графики, чекпойнты, MLflow

`PlotCallback` копит метрики по эпохам (хук `on_validation_end` — в этот момент
`logged_metrics` уже финализирован). MLflow-логгер подключается только если
сервер на `MLFLOW_TRACKING_URI` жив.

In [ ]:
from lightning.pytorch.callbacks import (
    ModelCheckpoint,
)

checkpoint_cb = ModelCheckpoint(
    dirpath=CKPT_DIR,
    filename="resnet18-{epoch:02d}-{val_acc:.3f}",
    monitor="val_acc",
    mode="max",
    save_top_k=1,
    save_last=True,
)


In [7]:
import socket
from urllib.parse import urlparse

from lightning.pytorch.loggers import MLFlowLogger


def _mlflow_reachable(uri: str, timeout: float = 2.0) -> bool:
    if not MLFLOW_ENABLED:
        return False
    parsed = urlparse(uri)
    try:
        with socket.create_connection((parsed.hostname, parsed.port), timeout=timeout):
            return True
    except OSError:
        return False


def make_logger():
    if not _mlflow_reachable(MLFLOW_TRACKING_URI):
        print(f"WARNING: MLflow недоступен ({MLFLOW_TRACKING_URI}) — работаем без логгера")
        return None
    return MLFlowLogger(
        tracking_uri=MLFLOW_TRACKING_URI,
        experiment_name=MLFLOW_EXPERIMENT_NAME,
        save_dir=str(ROOT / "mlruns"),
    )


## 6. Тренировка

60 эпох по умолчанию (для результата ближе к статье — 93-95% — подними
`EPOCHS` в ячейке конфигурации до 160-200).

In [ ]:
from lightning.pytorch import Trainer
from lightning.pytorch.profilers import AdvancedProfiler, PyTorchProfiler, SimpleProfiler


def make_profiler(name):
    if name in (None, "none"):
        return None
    if name == "simple":
        return SimpleProfiler(dirpath=str(CKPT_DIR), filename="profile")
    if name == "advanced":
        return AdvancedProfiler(dirpath=str(CKPT_DIR), filename="profile")
    if name == "pytorch":
        return PyTorchProfiler(dirpath=str(CKPT_DIR), filename="profile")
    raise ValueError(f"unknown profiler: {name}")

dm = CIFAR10DataModule(DATA_DIR, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
dm.prepare_data()  # качает CIFAR-10, если ещё нет

lit = ResNet18Lit(
    num_classes=10, lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY,
    warmup_epochs=WARMUP_EPOCHS, epochs=EPOCHS, min_lr=MIN_LR,
)

logger = make_logger()

trainer = Trainer(
    max_epochs=EPOCHS,
    accelerator="auto",
    devices="auto",
    callbacks=[checkpoint_cb],
    logger=logger,
    benchmark=True,
    precision="16-mixed",
    profiler=make_profiler(PROFILER),
    log_every_n_steps=20,
)

trainer.fit(lit, datamodule=dm)


a:\projects\spartan-torch\.venv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")
Using 16bit Automatic Mixed Precision (AMP)
Trainer already configured with model summary callbacks: [<class 'lightning.pytorch.callbacks.rich_model_summary.RichModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
a:\projects\spartan-torch\.venv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Py

┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model     │ Sequential         │ 11.2 M │ train │     0 │
│ 1 │ train_acc │ MulticlassAccuracy │      0 │ train │     0 │
│ 2 │ val_acc   │ MulticlassAccuracy │      0 │ train │     0 │
└───┴───────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 11.2 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 11.2 M                                                                                               
Total estimated model params size (MB): 44.727                                                                     
Modules in train mode: 84                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

a:\projects\spartan-torch\.venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, 
LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

a:\projects\spartan-torch\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 
'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.

a:\projects\spartan-torch\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 
'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.

`Trainer.fit` stopped: `max_epochs=65` reached.


🏃 View run upbeat-snail-229 at: http://localhost:5000/#/experiments/2/runs/3e3e2e6d2177446da90c19a70e97e1ea
🧪 View experiment at: http://localhost:5000/#/experiments/2


In [9]:
print(f"best ckpt: {checkpoint_cb.best_model_path}")
if logger is not None:
    print(f"MLflow run: {MLFLOW_TRACKING_URI}/#/experiments/{logger.experiment_id}/runs/{logger.run_id}")


best ckpt: A:\projects\spartan-torch\experiments\image_classification\resnet18\checkpoints\resnet18-epoch=64-val_acc=0.877.ckpt
MLflow run: http://localhost:5000/#/experiments/2/runs/3e3e2e6d2177446da90c19a70e97e1ea


## Итог

- `ResidualBlock` эквивалентен torchvision `BasicBlock` — проверено через
  `state_dict` и поэлементное сравнение форвардов.
- Эталонный пайплайн: pytorch-lightning + MLflow + inline-графики.
- Для точности из статьи подними `EPOCHS` до 160-200.
